# 10 - Intent Schema and Validation

## Objetivo
Demonstrar como uma intenção declarativa (YAML) é carregada, validada e
transformada em um modelo Pydantic tipado (`EntanglementIntent`) - e
mostrar, com exceções reais (não simuladas), tudo o que a validação
rejeita antes que qualquer coisa toque o SeQUeNCe.

## Conceitos
- `ibqn.intent.parser.load_intent_file`/`parse_intent_yaml`/`parse_intent_dict`
  convertem YAML/JSON/dict em `EntanglementIntent` - nunca há interpretação
  de linguagem natural (ver `docs/architecture.md`).
- `EntanglementIntent` é o WHAT: nunca especifica caminho, nó repetidor,
  ordem de swap, protocolo/rounds de purificação, memórias, regras ou
  algoritmo de roteamento (isso é o HOW, do `IntentPlanner` - ver notebook 12).
- `requirements.min_fidelity`/`requested_pairs` alimentam o **planejamento**
  (decidem se purificação é necessária, quantas memórias reservar);
  `validation.success_conditions` são **independentes** e definem o que a
  aplicação considera sucesso - os dois podem divergir (ver notebook 13).
- Toda validação usa `pydantic.ValidationError` real ou `ValueError`
  explícito dos parsers - nunca `eval()` para interpretar uma condição do
  YAML (`docs/assurance_design.md`).


## 3.1 Intent válido

Carregando o arquivo real `scenarios/intents/intent_001.yaml`.


In [1]:
from pathlib import Path

intent_path = Path("../scenarios/intents/intent_001.yaml")
print(intent_path.read_text(encoding="utf-8"))


intent:
  id: intent-001
  service: entanglement_distribution

  endpoints:
    source: A
    destination: B

  requirements:
    min_fidelity: 0.65
    min_throughput: 10
    max_latency: 0.5
    requested_pairs: 10
    start_time: 0.01
    duration: 0.05

  policy:
    priority: normal
    allow_rerouting: true
    allow_purification: true
    allow_multiple_paths: false

  validation:
    metrics:
      - delivered_pairs
      - average_fidelity
    success_conditions:
      delivered_pairs: ">= 10"
      average_fidelity: ">= 0.65"



In [2]:
from ibqn.intent.parser import load_intent_file

intent = load_intent_file(intent_path)
intent.id, intent.service, intent.endpoints.source, intent.endpoints.destination


2026-10-02 18:46:28,485 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'requested_pairs' -> 'reserved_memory_slots' (see docs/intent_resource_semantics.md)


2026-10-02 18:46:28,487 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'start_time' -> 'start_time_s' (see docs/intent_resource_semantics.md)


2026-10-02 18:46:28,488 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'duration' -> 'duration_s' (see docs/intent_resource_semantics.md)


('intent-001', 'entanglement_distribution', 'A', 'B')

## 3.2 Modelo tipado

Uma visão tabular (campo/valor) em vez de um dump interno do Pydantic.


In [3]:
from ibqn.demos.tables import intent_table

print(f"type(intent) = {type(intent).__name__}")
print(f"type(intent.endpoints) = {type(intent.endpoints).__name__}")
print(f"type(intent.requirements) = {type(intent.requirements).__name__}")
print(f"type(intent.validation) = {type(intent.validation).__name__}")
intent_table(intent)


type(intent) = EntanglementIntent
type(intent.endpoints) = IntentEndpoints
type(intent.requirements) = IntentRequirements
type(intent.validation) = IntentValidation


,field,value
0,id,intent-001
1,service,entanglement_distribution
2,source,A
3,destination,B
4,min_fidelity (dimensionless),0.65
5,min_throughput (pairs/s),10.0
6,max_latency (s),0.5
7,reserved_memory_slots,10
8,min_delivered_pairs,None
9,start_time_s (s),0.01


## 3.3 Conversão das condições

O YAML declara `success_conditions` como um mapeamento
`metric: "<operador> <valor>"`; `intent.parser._normalize_validation`
converte cada entrada em um objeto `SuccessCondition` via
`SuccessCondition.parse` - a mesma função usada abaixo diretamente.


In [4]:
from ibqn.intent.models import SuccessCondition

raw_yaml_conditions = {"delivered_pairs": ">= 10", "average_fidelity": ">= 0.90"}
parsed = {metric: SuccessCondition.parse(metric, expr) for metric, expr in raw_yaml_conditions.items()}
for metric, expr in raw_yaml_conditions.items():
    condition = parsed[metric]
    print(f"{metric!r}: {expr!r}  ->  SuccessCondition(metric={condition.metric!r}, operator={condition.operator!r}, expected={condition.expected})")


'delivered_pairs': '>= 10'  ->  SuccessCondition(metric='delivered_pairs', operator='>=', expected=10.0)
'average_fidelity': '>= 0.90'  ->  SuccessCondition(metric='average_fidelity', operator='>=', expected=0.9)


In [5]:
print("Conditions actually parsed from intent_001.yaml:")
for condition in intent.validation.success_conditions:
    print(f"  {condition}")


Conditions actually parsed from intent_001.yaml:
  delivered_pairs >= 10.0
  average_fidelity >= 0.65


## 3.4 Casos inválidos

Cada célula abaixo dispara **uma** violação real e imprime o tipo e a
mensagem da exceção - nenhuma é capturada de forma genérica.


In [6]:
from ibqn.intent.models import IntentEndpoints

try:
    IntentEndpoints(source="a", destination="a")
except Exception as e:
    print(f"{type(e).__name__}: {e}")


ValidationError: 1 validation error for IntentEndpoints
  Value error, source and destination must differ, both are 'a' [type=value_error, input_value={'source': 'a', 'destination': 'a'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [7]:
from ibqn.intent.models import IntentRequirements

try:
    IntentRequirements(min_fidelity=-0.1, min_throughput=1, max_latency=1, requested_pairs=1, start_time=0, duration=1)
except Exception as e:
    print(f"{type(e).__name__}: {e}")


2026-10-02 18:46:35,392 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'requested_pairs' -> 'reserved_memory_slots' (see docs/intent_resource_semantics.md)


2026-10-02 18:46:35,395 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'start_time' -> 'start_time_s' (see docs/intent_resource_semantics.md)


2026-10-02 18:46:35,396 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'duration' -> 'duration_s' (see docs/intent_resource_semantics.md)


ValidationError: 1 validation error for IntentRequirements
min_fidelity
  Input should be greater than or equal to 0 [type=greater_than_equal, input_value=-0.1, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal


In [8]:
try:
    IntentRequirements(min_fidelity=1.5, min_throughput=1, max_latency=1, requested_pairs=1, start_time=0, duration=1)
except Exception as e:
    print(f"{type(e).__name__}: {e}")


2026-10-02 18:46:35,427 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'requested_pairs' -> 'reserved_memory_slots' (see docs/intent_resource_semantics.md)


2026-10-02 18:46:35,429 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'start_time' -> 'start_time_s' (see docs/intent_resource_semantics.md)


2026-10-02 18:46:35,431 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'duration' -> 'duration_s' (see docs/intent_resource_semantics.md)


ValidationError: 1 validation error for IntentRequirements
min_fidelity
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


In [9]:
try:
    IntentRequirements(min_fidelity=0.5, min_throughput=1, max_latency=1, requested_pairs=1, start_time=0, duration=-1)
except Exception as e:
    print(f"{type(e).__name__}: {e}")


2026-10-02 18:46:35,453 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'requested_pairs' -> 'reserved_memory_slots' (see docs/intent_resource_semantics.md)


2026-10-02 18:46:35,455 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'start_time' -> 'start_time_s' (see docs/intent_resource_semantics.md)


2026-10-02 18:46:35,457 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'duration' -> 'duration_s' (see docs/intent_resource_semantics.md)


ValidationError: 1 validation error for IntentRequirements
duration_s
  Input should be greater than 0 [type=greater_than, input_value=-1, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than


In [10]:
try:
    IntentRequirements(min_fidelity=0.5, min_throughput=1, max_latency=1, requested_pairs=0, start_time=0, duration=1)
except Exception as e:
    print(f"{type(e).__name__}: {e}")


2026-10-02 18:46:35,476 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'requested_pairs' -> 'reserved_memory_slots' (see docs/intent_resource_semantics.md)


2026-10-02 18:46:35,478 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'start_time' -> 'start_time_s' (see docs/intent_resource_semantics.md)


2026-10-02 18:46:35,479 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'duration' -> 'duration_s' (see docs/intent_resource_semantics.md)


ValidationError: 1 validation error for IntentRequirements
reserved_memory_slots
  Input should be greater than 0 [type=greater_than, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than


In [11]:
try:
    SuccessCondition.parse("delivered_pairs", "~= 10")  # unsupported operator
except Exception as e:
    print(f"{type(e).__name__}: {e}")


ValueError: invalid success condition expression for metric 'delivered_pairs': '~= 10' (expected '<op> <value>' with op in >=, <=, >, <, ==, !=)

In [12]:
try:
    SuccessCondition.parse("delivered_pairs", ">= 10 and <= 20")  # composite expression, not a single comparison
except Exception as e:
    print(f"{type(e).__name__}: {e}")


ValueError: invalid success condition expression for metric 'delivered_pairs': '>= 10 and <= 20' (expected '<op> <value>' with op in >=, <=, >, <, ==, !=)


Note que operador desconhecido e expressão composta caem no **mesmo** `ValueError`: o parser só reconhece `<operador> <número>` via uma regex fixa - não há uma lista de operadores válidos separada da própria regex.


In [13]:
from ibqn.intent.models import IntentValidation

try:
    IntentValidation(
        metrics=["delivered_pairs"],
        success_conditions=[SuccessCondition(metric="average_fidelity", operator=">=", expected=0.9)],
    )
except Exception as e:
    print(f"{type(e).__name__}: {e}")


ValidationError: 1 validation error for IntentValidation
  Value error, success_conditions reference metrics not listed in 'metrics': ['average_fidelity'] [type=value_error, input_value={'metrics': ['delivered_p...or='>=', expected=0.9)]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [14]:
try:
    load_intent_file("../scenarios/intents/does_not_exist.yaml")
except Exception as e:
    print(f"{type(e).__name__}: {e}")


FileNotFoundError: [Errno 2] No such file or directory: '..\\scenarios\\intents\\does_not_exist.yaml'


In [15]:
from ibqn.intent.parser import parse_intent_yaml

try:
    parse_intent_yaml("intent: [this is not: valid: yaml")
except Exception as e:
    print(f"{type(e).__name__}: {e}")


ParserError: while parsing a flow sequence
  in "<unicode string>", line 1, column 9:
    intent: [this is not: valid: yaml
            ^
expected ',' or ']', but got ':'
  in "<unicode string>", line 1, column 28:
    intent: [this is not: valid: yaml
                               ^


## 3.5 Lifecycle

`IntentStatus` tem 13 estados; `intent.lifecycle.ALLOWED_TRANSITIONS`
declara explicitamente quais transições são permitidas a partir de cada
um - qualquer outra levanta `InvalidTransitionError`.


In [16]:
from ibqn.intent.models import IntentStatus

print(f"{len(list(IntentStatus))} possible states:")
print([status.value for status in IntentStatus])


13 possible states:
['RECEIVED', 'VALIDATED', 'REJECTED', 'PLANNING', 'PLANNED', 'DEPLOYING', 'ACTIVE', 'SATISFIED', 'VIOLATED', 'RECONCILING', 'FAILED', 'COMPLETED', 'CANCELLED']


In [17]:
from ibqn.intent.lifecycle import IntentLifecycle, InvalidTransitionError

lifecycle = IntentLifecycle("demo-lifecycle")
print(f"initial status: {lifecycle.status.value}")

lifecycle.transition(IntentStatus.VALIDATED, "schema validated")
print(f"valid transition RECEIVED -> VALIDATED: now {lifecycle.status.value}")

try:
    lifecycle.transition(IntentStatus.ACTIVE, "attempting to skip PLANNING/PLANNED/DEPLOYING")
except InvalidTransitionError as e:
    print(f"invalid transition rejected: {type(e).__name__}: {e}")


initial status: RECEIVED
valid transition RECEIVED -> VALIDATED: now VALIDATED
invalid transition rejected: InvalidTransitionError: intent 'demo-lifecycle' cannot transition from VALIDATED to ACTIVE; allowed: ['CANCELLED', 'PLANNING']


In [18]:
from ibqn.demos.tables import lifecycle_table

assert lifecycle.status == IntentStatus.VALIDATED, "the rejected transition above must not have changed the status"
lifecycle_table(lifecycle)


,from_status,to_status,reason,sim_time_s
0,(none),RECEIVED,intent received,None
1,RECEIVED,VALIDATED,schema validated,None


## Interpretação
O intent carregado do YAML real (`intent_001.yaml`) e o modelo tipado
resultante (`EntanglementIntent`) são o mesmo objeto que
`ibqn.execution.SequenceExecutor` consome nos notebooks seguintes - nada
aqui é reconstruído à parte. Cada exceção acima é a validação real do
Pydantic ou do parser (`intent.models`/`intent.parser`), não uma
simulação de erro escrita para este notebook. A transição inválida acima
não alterou `lifecycle.status`, confirmando que `IntentLifecycle.transition`
é atômica: ou a transição é aceita e registrada no histórico, ou nada
muda.

## Limitações
- O parser aceita apenas uma única comparação por métrica (`<op> <valor>`);
  expressões compostas (`E`/`OU`, faixas) não são suportadas nesta versão.
- Validação de schema (Pydantic) não verifica se `source`/`destination`
  existem numa topologia real - isso só é verificado quando
  `NetworkCapabilities`/`SequenceAdapter` tentam localizar o nó
  (`docs/architecture.md`).

## Conclusão
Intent demonstrado como um objeto formal, validável e independente de
qualquer configuração de rede - toda violação mostrada acima é detectada
antes que qualquer simulação seja construída.
